In [ ]:
# Importing libraries
from lumicks import pylake

import csv
import matplotlib.pyplot as plt
import os
import statistics


In [ ]:
def frame_tick_function(frame_ticks,offset):
    frame_tick_labels = []
    for frame_tick in frame_ticks:
        if frame_tick % 5 == 0:
            frame_tick_labels.append(frame_tick+offset)
        else:
            frame_tick_labels.append("")

    return frame_tick_labels

In [ ]:
# Setting parameters
root = '/Users/sc13967/Desktop/To process/'

crop_ranges = {'10':(0,37),'29':(14,35)}
crop_ranges = {}

# print(os.listdir(root))
for path in os.listdir(root):
    out_name, ext = os.path.splitext(root+path)

    # Only processing H5 files
    if (ext != ".h5"):
        continue
    
    # Loading data
    file = pylake.File(root+path)

    # Iterating over all scans in the file
    for scan in file.scans.values():        
        # Getting all timestamps in this file
        timestamps = file.downsampled_force1.timestamps

        # For each frame's time window, get all timestamps and calculate average force, etc.
        timestamp0 = scan.frame_timestamp_ranges()[0][0]
        results = []
        for (idx,(f_start,f_end)) in enumerate(scan.frame_timestamp_ranges()):
            f_timestamps = [timestamp > f_start and timestamp < f_end for timestamp in timestamps]

            force1 = statistics.mean(file.downsampled_force1.data[f_timestamps])
            force2 = statistics.mean(file.downsampled_force2.data[f_timestamps])
        
            results.append([idx,f_start,(f_start-timestamp0)/1E9,f_end,(f_end-timestamp0)/1E9,force1])
            
        # Plotting force as a function of time
        plt.figure()
        file.downsampled_force1.plot() 
        file.downsampled_force2.plot()
        
        with open(f"{out_name}_forces.csv", "w") as f:
            writer = csv.writer(f)
            writer.writerow(["Time (s)", "Force1 (pN)", "Force2 (pN)"])
            t = (file.downsampled_force1.timestamps-timestamp0)/1E9
            f1 = file.downsampled_force1.data
            f2 = file.downsampled_force2.data
            for row in zip(t,f1,f2):
                writer.writerow(row)
            
        # Cropping to preferred x range
        if scan.name in crop_ranges.keys():
            crop_range = crop_ranges[scan.name]
            ax1.set_xlim((results[crop_range[0]-1][2],results[crop_range[1]-1][2]))
            frame_ticks = range(crop_range[1]-crop_range[0]+1)
            frame_offset = crop_range[0]
        else:
            frame_ticks = range(len(scan.frame_timestamp_ranges()))
            frame_offset = 1
            
        plt.savefig(out_name+"_scan"+scan.name+"_forcetime.svg")
                        